In [8]:
import anndata as ad
import numpy as np
import scanpy as sc
import scanpy.external as sce

Remember to run the conversion before this

In [9]:
filePath = "../rawFiles/Prodzynski/converted_data.h5ad"
adata = sc.read_h5ad(filePath)

In [10]:
# Mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# Ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# Hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

This takes quite a while to run. Its grabbing annotations for changing from geneID to ensembl. Also, the gene annotations for this one is like a mixture of geneID and ensembl, for some reason.

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["ensembl_gene_id", "external_gene_name", "description"],
    host="www.ensembl.org"
)

annot = annot.drop_duplicates(subset=['external_gene_name'], keep='first')

symbol_to_ensembl = annot.set_index("external_gene_name")["ensembl_gene_id"]

In [12]:
adata.var['ensembl'] = adata.var.index.map(symbol_to_ensembl)
adata.var['geneID'] = adata.var_names
adata = adata[:, adata.var['ensembl'].notna()].copy()
adata.var_names = adata.var['ensembl']

In [13]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [14]:
adata.obs.rename(columns={"orig.ident" : "condition"}, inplace = True)
adata = adata[adata.obs["condition"] != "AK2087_organoid"]
adata.obs["condition"] = (adata.obs["condition"]).astype(str) + "_DAY15"
adata.obs["condition"] = adata.obs["condition"].astype("category")
adata.obs["Day"] = "DAY_15"
adata.obs["Day"] = adata.obs["Day"].astype("category")
adata.obs["source"] = "Prodzynski"
adata.obs["batch"] = adata.obs["old.ident"]
adata.obs["batch"] = adata.obs["batch"].astype(str).astype("category")
adata.obs["system"] = "hipsc"

<positron-console-cell-14>:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


In [15]:
adata.write_zarr("ProcessedData/ProdzynskiScanpy.zarr")

/home/kai/Documents/SCTLM-Bridges/.venv/lib/python3.14/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


In [16]:
adata = ad.read_zarr("ProcessedData/ProdzynskiScanpy.zarr")